# Projeto Praso — Previsão de Inadimplência
## EDA: Dados Comportamentais de Pedidos (`credito_comportamental_pedidos_final.csv`)

### 📌 Propósito deste Notebook
> **Nota de Contexto**: Este bloco introdutório apresenta a contextualização e o objetivo das análises preliminares da base de pedidos. Ao término das análises, esta seção será atualizada com um **Sumário Executivo** consolidando os padrões comportamentais de pagamento e recompra.

O objetivo deste notebook é realizar a **Análise Exploratória de Dados (EDA)** sobre o histórico transacional dos varejistas na Praso. Esse conjunto é a matéria-prima do **Modelo Comportamental (Clientes Recorrentes)**, permitindo responder: *como o histórico de volume de pedidos, pontualidade no pagamento e tickets médios refinam a estimativa de risco de inadimplência de clientes ativos?*

---

### 🧱 Separação e Dicionário das Features Brutas
O dataset de pedidos contém **5 variáveis brutas**:

1. `id_pedido`: Identificador unívoco da transação/pedido.
2. `id_cliente`: Identificador do cliente que realizou a compra (chave de ligação com a tabela de clientes).
3. `valor`: Faixa de valor monetário das mercadorias adquiridas (intervalar, ex.: `(0, 500]`, `(500, 1000]`), líquido de cupons.
4. `atraso`: Faixa de dias de atraso na liquidação do pagamento em relação ao prazo estipulado (intervalar, ex.: `(-100, 0]`, `(0, 2]`, `(2, 5]`). Valores $\le 0$ indicam pagamento antecipado ou pontual.
5. `data_entrega`: Registro cronológico da data de entrega das mercadorias pela logística da Praso (formato temporal `AAAA-MM-DD`).


In [ ]:
# 1. Configuração do Ambiente e Importação de Bibliotecas
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 10
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.3f' % x)

print("Ambiente configurado com sucesso!")


### 1. Compreensão Estrutural e Carga dos Dados de Pedidos


In [ ]:
# Carga da base comportamental
df_pedidos = pd.read_csv('data/datasets/credito_comportamental_pedidos_final.csv')

n_pedidos, n_colunas_ped = df_pedidos.shape
n_clientes_com_pedidos = df_pedidos['id_cliente'].nunique()
n_duplicatas_ped = df_pedidos.duplicated().sum()

print(f"Dimensões do Dataset de Pedidos: {n_pedidos} registros x {n_colunas_ped} colunas")
print(f"Total de Clientes Distintos com Pedidos: {n_clientes_com_pedidos}")
print(f"Linhas Duplicadas Exatas: {n_duplicatas_ped}")

df_pedidos.head()


### 2. Mapeamento de Tipos e Valores Faltantes


In [ ]:
# Resumo de tipos e nulos na tabela de pedidos
info_missing_ped = pd.DataFrame({
    'Tipo Dtype': df_pedidos.dtypes,
    'Valores Preenchidos': df_pedidos.notnull().sum(),
    'Qtd Faltantes': df_pedidos.isnull().sum(),
    '% Faltantes': (df_pedidos.isnull().sum() / len(df_pedidos) * 100).round(2),
    'Valores Únicos': df_pedidos.nunique()
})

display(info_missing_ped)


### 3. Distribuição de Pedidos por Cliente (Frequência Transacional)
Análise de quantos pedidos cada cliente realizou no período disponível.


In [ ]:
# Contagem de pedidos por cliente
pedidos_por_cliente = df_pedidos.groupby('id_cliente').size()

print("Estatísticas descritivas do volume de pedidos por cliente:")
display(pedidos_por_cliente.describe())

plt.figure(figsize=(10, 5))
sns.histplot(pedidos_por_cliente, bins=20, kde=False, color='#1976d2')
plt.title('Distribuição da Quantidade de Pedidos por Cliente', fontsize=13, fontweight='bold')
plt.xlabel('Número de Pedidos Realizados')
plt.ylabel('Frequência de Clientes')
plt.tight_layout()
plt.show()


### 4. Distribuição das Faixas de Atraso e Valores dos Pedidos


In [ ]:
# Distribuição das faixas de atraso e faixas de valor
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Faixas de atraso
ordem_atraso = df_pedidos['atraso'].value_counts().index
sns.countplot(data=df_pedidos, y='atraso', order=ordem_atraso, ax=axes[0], palette='Blues_r')
axes[0].set_title('Distribuição das Faixas de Atraso nos Pedidos', fontweight='bold')
axes[0].set_xlabel('Contagem de Pedidos')
axes[0].set_ylabel('Faixa de Atraso')

# Faixas de valor
ordem_valor = df_pedidos['valor'].value_counts().index
sns.countplot(data=df_pedidos, y='valor', order=ordem_valor, ax=axes[1], palette='Purples_r')
axes[1].set_title('Distribuição das Faixas de Valor dos Pedidos', fontweight='bold')
axes[1].set_xlabel('Contagem de Pedidos')
axes[1].set_ylabel('Faixa de Valor')

plt.tight_layout()
plt.show()
